In [1]:
import pandas as pd
import numpy as np
import requests
import re
import matplotlib.pyplot as plt
import seaborn as sns
from bs4 import BeautifulSoup
import json

Checked robots.txt of 247sports and I'm good to scrape the public-facing pages

In [6]:
jc_rankings_url = 'https://247sports.com/college/middle-tennessee-state/season/2023-football/compositerecruitrankings/?InstitutionGroup=JuniorCollege'

headers = {
    "User-Agent": "Mozilla/5.0"
}

jc_response = requests.get(jc_rankings_url, headers=headers)
jc_soup = BeautifulSoup(jc_response.text, "html.parser")

print(jc_soup.prettify())

<!DOCTYPE html>
<html class="twofourseven-site" data-template="responsive" lang="en-US" xmlns="http://www.w3.org/1999/xhtml">
 <head>
  <meta charset="utf-8"/>
  <meta content="IE=edge,chrome=1" http-equiv="X-UA-Compatible"/>
  <meta content="width=device-width, initial-scale=1" name="viewport"/>
  <meta content="telephone=no" name="format-detection">
   <meta content="247Sports" name="application-name">
    <meta content="max-image-preview:large" name="robots">
     <meta content="#004B82" name="msapplication-TileColor"/>
     <meta content="https://s3media.247sports.com/Content/Img/logo_win8.png" name="msapplication-TileImage"/>
     <title>
      2023 Top Football Recruits
     </title>
     <link href="https://s3media.247sports.com/Content/Img/247touch.png" rel="apple-touch-icon"/>
     <link href="https://s3media.247sports.com/favicon.ico" rel="shortcut icon" type="image/x-icon"/>
     <link href="https://cdn.cookielaw.org" rel="preconnect"/>
     <link href="https://geolocation.o

Did this in a very similar way to how I did my data acquisition project

In [10]:
players = jc_soup.find_all("li", class_="rankings-page__list-item")

data = {'First': [], 'Last': [], 'Height': [], 'Weight': [], 'Score': []}

def create_data(players):
    for player in players:
        name_tag = player.find('a', class_='rankings-page__name-link')
        if not name_tag:
            continue
        name = name_tag.text.strip().split()
        first = name[0]
        last = name[-1]
        metrics = player.find('div', class_='metrics')
        if metrics:
            hw = metrics.text.strip().split('/')
            height = hw[0].strip()
            weight = hw[1].strip()
        else:
            height, weight = None, None
        score_tag = player.find('span', class_='score')
        score = score_tag.text.strip() if score_tag else None
        data['First'].append(first)
        data['Last'].append(last)
        data['Height'].append(height)
        data['Weight'].append(weight)
        data['Score'].append(score)

create_data(players)

df_jc = pd.DataFrame(data)
df_jc.head()

,First,Last,Height,Weight,Score
0,Malik,Benson,6-1,185,0.9431
1,Justin,Jefferson,6-1,215,0.9208
2,Elijah,Davis,6-4,280,0.9061
3,George,Silva,6-7,295,0.9003
4,Channing,Canada,6-0,185,0.9000
